# walkman: basic dashboard

Launched by `walkman start <instrument>`. Run all cells, then fill in the inputs and click **Run**.

In [1]:
%matplotlib widget
import os
from html import escape
import threading
from pathlib import Path
import pandas as pd

import ipywidgets as widgets
from IPython.display import display

from walkman.instruments import CONFIG_ENV_VAR, INSTRUMENT_ENV_VAR, build_image_creator, load_instrument
from walkman.noise import NoisePSD, NoiseResult
from walkman.plots import (HistogramGrid, NoiseTrendPlot, PSDPlot, noise_label_html, noise_value_html,
                           series_color)

missing = [var for var in (INSTRUMENT_ENV_VAR, CONFIG_ENV_VAR) if var not in os.environ]
if missing:
    raise KeyError(f"Environment variables {missing} are not set. Launch this notebook with `walkman start <instrument>`.")

instrument_module = load_instrument(os.environ[INSTRUMENT_ENV_VAR])
creator = build_image_creator(instrument_module, os.environ[CONFIG_ENV_VAR], max_batch_size=1)
noisetask = NoisePSD()

2026-09-23 11:33:53,792 - DetectorConfig - INFO - Config loaded from file '../instruments/KPF/kpf.yaml'


In [2]:
wide = {"description_width": "initial"}
input_path = widgets.Text(description="Data folder", placeholder="absolute/path/to/data", style=wide, layout=widgets.Layout(width="60%"))
watch_mode = widgets.Checkbox(value=False, description="Watch mode", indent=False)
poll_interval = widgets.BoundedFloatText(value=5.0, min=0.0, max=3600.0, step=0.5, description="Poll interval (s)",
                                         style=wide, disabled=True)
# noise analysis inputs
region_boxes = {region: widgets.Checkbox(value=region in ("overscan"), description=region, indent=False)
                for region in ("full", "image", "overscan")}
trim_start = widgets.BoundedIntText(value=0, min=0, max=100000, description="Trim start", style=wide)
trim_end = widgets.BoundedIntText(value=0, min=0, max=100000, description="Trim end", style=wide)
target_shape = widgets.BoundedIntText(value=0, min=0, max=100000, description="Target shape (0 = auto)", style=wide)
window_func = widgets.Text(value="hamming", description="Window function", style=wide)
run_button = widgets.Button(description="Run", button_style="primary", icon="play")
status = widgets.HTML()

# poll interval only matters in watch mode
widgets.dlink((watch_mode, "value"), (poll_interval, "disabled"), transform=lambda watch: not watch)

# dashboard widgets (noise table, combined PSD, noise trend, histogram grid), refreshed every lazy_run batch.
# result_rows maps (det_id, output, region) to its color, table widgets and latest results, in order of first appearance.
result_rows = {}
table_header = [widgets.HTML(f"<b style='font-size:14px;color:#52514e'>{name}</b>") for name in ("", "Noise", "PSD")]
noise_table = widgets.GridBox(table_header, layout=widgets.Layout(
    width="50%", grid_template_columns="1fr auto auto", grid_gap="4px 16px", align_items="center",
    max_height="400px", overflow_y="auto"))
psd_plot = PSDPlot()
trend_plot = NoiseTrendPlot(xlabel=f"Time ({noisetask.meta['timestamp_key']})")
hist_grid = HistogramGrid()
dashboard = widgets.VBox([widgets.HBox([noise_table, psd_plot.widget]), trend_plot.widget, hist_grid.widget],
                         layout=widgets.Layout(display="none"))

def _add_table_row(key, color):
    noise_w = widgets.HTML()
    show = widgets.Checkbox(value=True, indent=False, layout=widgets.Layout(width="auto"))

    def _toggle(change):
        psd_plot.set_visible(key, change["new"])
        trend_plot.set_visible(key, change["new"])

    show.observe(_toggle, names="value")
    noise_table.children = (*noise_table.children, widgets.HTML(noise_label_html(*key, color)), noise_w, show)
    return noise_w

def _update_rows(noise_df):
    for rec in noise_df.itertuples(index=False):
        key = (rec.det_id, rec.output, rec.region)
        if key not in result_rows:
            # colors follow the row identity, so they stay fixed across refreshes
            color = series_color(len(result_rows))
            result_rows[key] = {"color": color, "noise_w": _add_table_row(key, color)}
        row = result_rows[key]
        row.update(psd=rec.psd, histogram=rec.histogram)
        row["noise_w"].value = noise_value_html(rec.noise, row["color"])
    psd_plot.update({key: (row["color"], *row["psd"]) for key, row in result_rows.items()})
    # the trend uses the accumulated result.data, so it shows every image processed so far
    trend_plot.update({key: (result_rows[key]["color"], group["time"].astype(float), group["noise"])
                       for key, group in result.data.groupby(["det_id", "output", "region"], sort=False)
                       if key in result_rows})
    hist_grid.update({key: (" · ".join(map(str, key)), row["color"], *row["histogram"])
                      for key, row in result_rows.items()})
    dashboard.layout.display = None

def _clear_rows():
    result_rows.clear()
    noise_table.children = table_header
    psd_plot.clear()
    trend_plot.clear()
    hist_grid.clear()
    dashboard.layout.display = "none"
    result = NoiseResult.model_construct()  # empty result to accumulate all images processed so far

def _run(input_path, watch, interval, region_boxes, trim_start, trim_end, target_shape, window_func):
    # Runs in a background thread so the kernel stays responsive
    path = Path(input_path.value.strip()).expanduser().resolve()
    if not path.is_absolute():
        status.value = "<b style='color:red'>Error:</b> path must be absolute."
        return
    path = str(path)

    regions = [region for region, box in region_boxes.items() if box.value]
    if not regions:
        status.value = "<b style='color:red'>Error:</b> select at least one region."
        return

    global result
    _clear_rows()
    try:
        creator.watch_mode = watch
        creator.poll_interval = interval
        for imres in creator.lazy_run(input_source=path):
            bundle = imres.data
            if len(bundle) > 0:
                noiseres = noisetask.run(bundle, regions=regions, trim_start=trim_start.value, trim_end=trim_end.value,
                                         target_shape=target_shape.value or None, window_func=window_func.value or 'hamming')
                result = result.combine(noiseres)
                _update_rows(noiseres.data)
            else:
                status.value += "<br>No data to analyze."
        status.value += "<br>Analysis complete."
    except Exception as e:
        # Exceptions in threads are otherwise lost, so surface them in the UI
        status.value = f"<b style='color:red'>Error:</b> {type(e).__name__}: {e}"
    finally:
        run_button.disabled = False

def on_run(_):
    run_button.disabled = True
    status.value = f"Watching every {poll_interval.value} s..." if watch_mode.value else "Running..."
    threading.Thread(target=_run,
                     args=(input_path, watch_mode.value, poll_interval.value, region_boxes, trim_start, trim_end,
                           target_shape, window_func),
                     daemon=True).start()

result = NoiseResult.model_construct()  # empty result to accumulate all images processed so far
run_button.on_click(on_run)
display(widgets.VBox([
    input_path,
    widgets.HBox([watch_mode, poll_interval]),
    widgets.HBox([widgets.Label("Regions:"), *region_boxes.values()]),
    widgets.HBox([trim_start, trim_end]),
    widgets.HBox([target_shape, window_func]),
    run_button,
    status,
    dashboard,
]))